In [0]:
# Databricks notebook source
# Studio 3 - Rischio mensile di bassa produzione rinnovabile
#
# Qui imposto:
# - librerie;
# - catalogo;
# - tabella Gold finale;
# - controllo iniziale sui dati;
# - funzioni comuni per lettura dati, grafici e conversioni.

# Importo le librerie principali.
import pandas as pd
import plotly.express as px


# Imposto il catalogo e la tabella Gold finale.
catalogo = "progetto_meteo"
tabella_dati_studio = f"{catalogo}.gold.dati_studio"


# Seleziono il catalogo del progetto.
spark.sql(f"USE CATALOG {catalogo}")


# Controllo che la Gold finale sia popolata.
# Se la tabella è vuota, i casi studio non possono essere eseguiti.
righe_gold = spark.table(tabella_dati_studio).count()

if righe_gold == 0:
    raise Exception(
        f"La tabella {tabella_dati_studio} è vuota. "
        "Prima esegui Gold_Dati_Aggregati e Gold_Dati_Studio."
    )

print(f"Tabella pronta: {tabella_dati_studio}")
print(f"Righe disponibili: {righe_gold}")


# Leggo una query direttamente da Databricks.
# Dentro Databricks il notebook può usare Spark senza credenziali locali.
def leggi_databricks(query):

    return spark.sql(query).toPandas()


# Mostro i grafici Plotly dentro Databricks.
# displayHTML rende il grafico più stabile nei notebook Databricks.
def mostra_grafico(fig):

    displayHTML(
        fig.to_html(
            include_plotlyjs=True,
            full_html=False,
            config={"responsive": True}
        )
    )


# Converto colonne pandas in numerico quando serve.
# Questo evita problemi di visualizzazione con Plotly.
def converti_colonne_numeriche(df, colonne):

    for colonna in colonne:
        if colonna in df.columns:
            df[colonna] = pd.to_numeric(df[colonna], errors="coerce")

    return df

In [0]:
# Studio 3 - Rischio mensile di bassa produzione rinnovabile
#
# Questo studio misura quanto spesso una macroarea entra in una fase
# di produzione rinnovabile giornaliera bassa.
#
# Per ogni area viene calcolata la produzione giornaliera totale:
# Solare + Eolico + Idrico.
#
# Poi, per ogni macroarea, viene calcolata una soglia bassa usando
# il primo quartile della produzione giornaliera.
#
# Un giorno viene considerato critico quando la sua produzione è inferiore
# alla soglia bassa della rispettiva area.
#
# Il risultato finale è una heatmap:
# - righe = macroaree;
# - colonne = mesi;
# - valore = percentuale di giorni critici nel mese.
#
# L'obiettivo è individuare i mesi e le aree in cui la produzione rinnovabile
# mostra più frequentemente valori deboli rispetto al comportamento tipico
# della stessa macroarea.
#
# In ambito gestionale, questa analisi può aiutare a evidenziare periodi
# in cui potrebbe servire maggiore attenzione alla copertura energetica,
# alla programmazione della capacità o alla necessità di fonti di compensazione.
#
# In ambito economico o di investimento, può essere usata per ragionare
# sulla regolarità della produzione rinnovabile, sulla vulnerabilità stagionale
# delle diverse macroaree e sulla stabilità relativa del mix energetico.


# Imposto l'anno da analizzare.
anno_studio = 2025


# Leggo la produzione giornaliera per area dalla Gold finale.
query_rischio = f"""
SELECT
    Area,
    to_date(Data, 'dd/MM/yyyy') AS Data,
    month(to_date(Data, 'dd/MM/yyyy')) AS Mese_Num,
    CAST(
        ROUND(
            SUM(Solare + Eolico + Idrico) / 1000000,
            4
        ) AS DOUBLE
    ) AS Produzione_Giornaliera
FROM {tabella_dati_studio}
WHERE year(to_date(Data, 'dd/MM/yyyy')) = {anno_studio}
GROUP BY
    Area,
    to_date(Data, 'dd/MM/yyyy'),
    month(to_date(Data, 'dd/MM/yyyy'))
ORDER BY
    Area,
    Data
"""

df_rischio_base = leggi_databricks(query_rischio)


# Converto le colonne numeriche.
df_rischio_base = converti_colonne_numeriche(
    df_rischio_base,
    [
        "Mese_Num",
        "Produzione_Giornaliera"
    ]
)


# Controllo i dati letti.
display(df_rischio_base)

print("Righe lette:", len(df_rischio_base))


# Creo il nome mese in italiano.
nomi_mesi = {
    1: "Gen",
    2: "Feb",
    3: "Mar",
    4: "Apr",
    5: "Mag",
    6: "Giu",
    7: "Lug",
    8: "Ago",
    9: "Set",
    10: "Ott",
    11: "Nov",
    12: "Dic"
}

df_rischio_base["Mese"] = df_rischio_base["Mese_Num"].map(nomi_mesi)


# Calcolo la soglia bassa per ogni area.
# Uso il primo quartile: sotto questa soglia considero il giorno critico.
df_rischio_base["Soglia_Bassa"] = (
    df_rischio_base
    .groupby("Area")["Produzione_Giornaliera"]
    .transform(lambda valori: valori.quantile(0.25))
)


# Identifico i giorni critici.
df_rischio_base["Giorno_Critico"] = (
    df_rischio_base["Produzione_Giornaliera"] < df_rischio_base["Soglia_Bassa"]
)


# Calcolo giorni critici e percentuale per mese e area.
df_rischio = (
    df_rischio_base
    .groupby(
        [
            "Area",
            "Mese_Num",
            "Mese"
        ],
        as_index=False
    )
    .agg(
        Giorni_Totali=("Giorno_Critico", "count"),
        Giorni_Critici=("Giorno_Critico", "sum")
    )
)

df_rischio["Percentuale_Critica"] = (
    df_rischio["Giorni_Critici"] / df_rischio["Giorni_Totali"] * 100
).round(0)


# Ordino mesi e aree.
ordine_mesi = [
    "Gen",
    "Feb",
    "Mar",
    "Apr",
    "Mag",
    "Giu",
    "Lug",
    "Ago",
    "Set",
    "Ott",
    "Nov",
    "Dic"
]

ordine_aree = [
    "Nord",
    "Centro",
    "Sud",
    "Isole"
]

df_rischio["Mese"] = pd.Categorical(
    df_rischio["Mese"],
    categories=ordine_mesi,
    ordered=True
)

df_rischio["Area"] = pd.Categorical(
    df_rischio["Area"],
    categories=ordine_aree,
    ordered=True
)

df_rischio = df_rischio.sort_values(["Area", "Mese_Num"])


# Controllo la tabella finale del rischio.
display(df_rischio)


# Creo la tabella delle percentuali per la heatmap.
tabella_percentuali = (
    df_rischio
    .pivot(
        index="Area",
        columns="Mese",
        values="Percentuale_Critica"
    )
    .fillna(0)
)


# Creo la tabella dei giorni critici.
# Serve per costruire il dettaglio del tooltip.
tabella_giorni = (
    df_rischio
    .pivot(
        index="Area",
        columns="Mese",
        values="Giorni_Critici"
    )
    .fillna(0)
)


# Creo la tabella dei giorni totali.
# Serve per mostrare il rapporto giorni critici / giorni disponibili.
tabella_totali = (
    df_rischio
    .pivot(
        index="Area",
        columns="Mese",
        values="Giorni_Totali"
    )
    .fillna(0)
)


# Creo il testo visibile dentro le celle.
testo_celle = tabella_percentuali.astype(int).astype(str) + "%"


# Creo il testo dettagliato per il tooltip.
testo_hover = []

for area in tabella_percentuali.index:

    riga_hover = []

    for mese in tabella_percentuali.columns:

        percentuale = tabella_percentuali.loc[area, mese]
        giorni_critici = int(tabella_giorni.loc[area, mese])
        giorni_totali = int(tabella_totali.loc[area, mese])

        riga_hover.append(
            f"Area: {area}<br>"
            f"Mese: {mese} {anno_studio}<br>"
            f"Giorni critici: {giorni_critici}/{giorni_totali}<br>"
            f"Rischio: {percentuale:.0f}%"
        )

    testo_hover.append(riga_hover)


# Creo una scala colori dal bianco panna al rosso scuro.
# I valori più chiari indicano una quota minore di giorni critici.
# I valori più scuri indicano una quota maggiore di giorni critici.
scala_rischio = [
    [0.00, "#FFF8DC"],
    [0.25, "#FDD49E"],
    [0.50, "#FC8D59"],
    [0.75, "#D7301F"],
    [1.00, "#7F0000"]
]


# Creo la heatmap del rischio.
fig = px.imshow(
    tabella_percentuali,
    text_auto=False,
    aspect="auto",
    color_continuous_scale=scala_rischio,
    zmin=0,
    zmax=100,
    title=f"Rischio mensile di bassa produzione rinnovabile - {anno_studio}",
    labels={
        "x": "Mese",
        "y": "Macroarea",
        "color": "Rischio"
    }
)


# Mostro le percentuali dentro le celle e il dettaglio nel tooltip.
fig.update_traces(
    text=testo_celle.values,
    texttemplate="%{text}",
    hovertext=testo_hover,
    hovertemplate="%{hovertext}<extra></extra>"
)


# Sistemo il layout.
fig.update_layout(
    height=550,
    title_x=0.5,
    coloraxis_colorbar=dict(
        title="% giorni<br>critici"
    )
)


# Mostro il grafico.
mostra_grafico(fig)